# Vollständiges Multi-Algorithmus Hyperparameter-Tuning

## Ausführliche Version für Cloud-Server

**Algorithmen:**
- NSGA-II
- SMS-EMOA  
- NSGA-III

**Features:**
- Umfangreiche Hyperparameter-Suche
- Mehrere Evaluations-Seeds für Robustheit
- Detaillierte Analyse und Visualisierung

---

## 1. Setup

In [ ]:
import numpy as np
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))
import json
import warnings
import multiprocessing as mp
warnings.filterwarnings('ignore')

# Matplotlib Style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    try:
        plt.style.use('seaborn-whitegrid')
    except:
        pass

plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12

print(f"Python Hyperparameter Tuning - Full Version")
print(f"Available CPU cores: {mp.cpu_count()}")

---

## 2. Konfiguration

In [ ]:
# Verzeichnisse
RESULTS_DIR = Path("../results")
PLOTS_DIR = Path("../plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# Problem-Parameter
try:
    with open(RESULTS_DIR / 'config.json', 'r') as f:
        CONFIG = json.load(f)
    SEED = CONFIG['seed']
    K = CONFIG['K']
    L = CONFIG['L']
    XL = CONFIG['xl']
    XU = CONFIG['xu']
    print(f"Config loaded: K={K}, L={L}")
except FileNotFoundError:
    print("config.json not found, using defaults")
    SEED = 42
    K = 22
    L = 4
    XL = 1.0
    XU = 10.0

In [ ]:
# ============================================================
# VOLLSTÄNDIGE TUNING PARAMETER
# ============================================================

N_TRIALS = 40                # Trials pro Algorithmus (40 x 3 = 120 total)
N_GEN = 100                  # Generationen pro Trial
R_SIMULATIONS = 1000         # Simulationen pro Evaluation
EVAL_SEEDS = [42, 101, 202, 303, 404]  # 5 Seeds für robuste Bewertung

# Algorithmen
ALGORITHMS = ['NSGA-II', 'SMS-EMOA', 'NSGA-III']

# Erweiterte Hyperparameter-Suchräume
SEARCH_SPACE = {
    'pop_size': (50, 250, 10),        # 50 bis 250 in 10er Schritten
    'eta_crossover': (5.0, 30.0),     # SBX eta
    'eta_mutation': (5.0, 30.0),      # PM eta
    'crossover_prob': (0.6, 1.0),     # Crossover Wahrscheinlichkeit
    'mutation_prob': (None, None),    # None = 1/n_var (default)
    'n_partitions': (6, 20),          # Für NSGA-III
}

print("=" * 70)
print("FULL HYPERPARAMETER TUNING CONFIGURATION")
print("=" * 70)
print(f"Algorithms: {ALGORITHMS}")
print(f"Trials per Algorithm: {N_TRIALS}")
print(f"Total Trials: {len(ALGORITHMS) * N_TRIALS}")
print(f"Generations per Trial: {N_GEN}")
print(f"Simulations per Evaluation: {R_SIMULATIONS}")
print(f"Evaluation Seeds: {EVAL_SEEDS} ({len(EVAL_SEEDS)} seeds)")
print("=" * 70)
print("\nSearch Space:")
for param, values in SEARCH_SPACE.items():
    print(f"  {param}: {values}")
print("=" * 70)

# Geschätzte Laufzeit
est_time_per_trial = N_GEN * 0.5  # ~0.5 Sekunden pro Generation (grobe Schätzung)
est_time_per_trial *= len(EVAL_SEEDS)  # Mal Anzahl Seeds
est_total_time = len(ALGORITHMS) * N_TRIALS * est_time_per_trial / 60
print(f"\nGeschätzte Laufzeit: {est_total_time:.0f} - {est_total_time*2:.0f} Minuten")

---

## 3. Trial-Funktion

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

def run_single_trial(params):
    """
    Führt einen einzelnen Hyperparameter-Trial aus.
    Evaluiert über mehrere Seeds für robuste Ergebnisse.
    """
    import numpy as np
    import random
    import time
    
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.moo.nsga3 import NSGA3
    from pymoo.algorithms.moo.sms import SMSEMOA
    from pymoo.operators.crossover.sbx import SBX
    from pymoo.operators.mutation.pm import PM
    from pymoo.operators.sampling.rnd import FloatRandomSampling
    from pymoo.optimize import minimize
    from pymoo.termination import get_termination
    from pymoo.indicators.hv import HV
    from pymoo.util.ref_dirs import get_reference_directions
    from simulation import TopTrumpsSimulation, TopTrumpsBalancing
    
    # Parameter extrahieren
    trial_id = params['trial_id']
    algorithm_name = params['algorithm']
    pop_size = params['pop_size']
    eta_crossover = params['eta_crossover']
    eta_mutation = params['eta_mutation']
    crossover_prob = params['crossover_prob']
    K = params['K']
    L = params['L']
    XL = params['XL']
    XU = params['XU']
    R_SIMS = params['R_SIMULATIONS']
    N_GEN = params['N_GEN']
    EVAL_SEEDS = params['EVAL_SEEDS']
    
    # Simulation erstellen
    sim = TopTrumpsSimulation(num_cards=K, num_categories=L)
    problem = TopTrumpsBalancing(sim, n_simulations=R_SIMS, xl=XL, xu=XU)
    termination = get_termination("n_gen", N_GEN)
    
    # Algorithmus erstellen
    def create_algorithm():
        if algorithm_name == 'NSGA-II':
            return NSGA2(
                pop_size=pop_size,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
        elif algorithm_name == 'SMS-EMOA':
            return SMSEMOA(
                pop_size=pop_size,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
        elif algorithm_name == 'NSGA-III':
            n_partitions = params.get('n_partitions', 12)
            ref_dirs = get_reference_directions("das-dennis", 2, n_partitions=n_partitions)
            return NSGA3(
                pop_size=pop_size,
                ref_dirs=ref_dirs,
                sampling=FloatRandomSampling(),
                crossover=SBX(prob=crossover_prob, eta=eta_crossover),
                mutation=PM(eta=eta_mutation),
                eliminate_duplicates=True
            )
    
    # Evaluation über mehrere Seeds
    hvs = []
    n_solutions = []
    start_time = time.time()
    
    for eval_seed in EVAL_SEEDS:
        np.random.seed(eval_seed)
        random.seed(eval_seed)
        
        try:
            algorithm = create_algorithm()
            res = minimize(problem, algorithm, termination, seed=eval_seed, verbose=False)
            
            # Hypervolume berechnen (Referenzpunkt [0,0] da wir maximieren)
            hv = HV(ref_point=np.array([0.0, 0.0]))(res.F)
            hvs.append(hv)
            n_solutions.append(len(res.F))
        except Exception as e:
            hvs.append(float('nan'))
            n_solutions.append(0)
    
    runtime = time.time() - start_time
    
    # Ergebnis zusammenstellen
    result = {
        'trial_id': trial_id,
        'algorithm': algorithm_name,
        'pop_size': pop_size,
        'eta_crossover': eta_crossover,
        'eta_mutation': eta_mutation,
        'crossover_prob': crossover_prob,
        'hypervolume_mean': float(np.nanmean(hvs)),
        'hypervolume_std': float(np.nanstd(hvs)),
        'hypervolume_min': float(np.nanmin(hvs)),
        'hypervolume_max': float(np.nanmax(hvs)),
        'n_solutions_mean': float(np.mean(n_solutions)),
        'runtime_seconds': runtime,
        'status': 'completed' if not np.isnan(np.nanmean(hvs)) else 'failed'
    }
    
    if algorithm_name == 'NSGA-III':
        result['n_partitions'] = params.get('n_partitions')
    
    return result

---

## 4. Trial-Konfigurationen generieren

In [ ]:
def generate_trial_params(algorithm, trial_id, seed):
    """Generiert zufällige Hyperparameter aus dem Suchraum."""
    np.random.seed(seed)
    
    pop_min, pop_max, pop_step = SEARCH_SPACE['pop_size']
    
    params = {
        'trial_id': trial_id,
        'algorithm': algorithm,
        'pop_size': int(np.random.choice(range(pop_min, pop_max + 1, pop_step))),
        'eta_crossover': np.random.uniform(*SEARCH_SPACE['eta_crossover']),
        'eta_mutation': np.random.uniform(*SEARCH_SPACE['eta_mutation']),
        'crossover_prob': np.random.uniform(*SEARCH_SPACE['crossover_prob']),
        'K': K,
        'L': L,
        'XL': XL,
        'XU': XU,
        'R_SIMULATIONS': R_SIMULATIONS,
        'N_GEN': N_GEN,
        'EVAL_SEEDS': EVAL_SEEDS,
    }
    
    if algorithm == 'NSGA-III':
        params['n_partitions'] = int(np.random.randint(*SEARCH_SPACE['n_partitions']))
    
    return params


# Alle Trials generieren
all_trials = []
trial_id = 0

for algo in ALGORITHMS:
    for i in range(N_TRIALS):
        params = generate_trial_params(algo, trial_id, SEED + trial_id)
        all_trials.append(params)
        trial_id += 1

print(f"Generated {len(all_trials)} trial configurations")
print(f"\nBeispiel-Konfiguration (Trial 0):")
example = all_trials[0]
for k, v in example.items():
    if k not in ['EVAL_SEEDS', 'K', 'L', 'XL', 'XU', 'R_SIMULATIONS', 'N_GEN']:
        print(f"  {k}: {v}")

---

## 5. Sequentielle Ausführung

In [ ]:
print("=" * 70)
print("STARTING FULL HYPERPARAMETER TUNING")
print("=" * 70)
print(f"Total Trials: {len(all_trials)}")
print(f"Eval Seeds per Trial: {len(EVAL_SEEDS)}")
print(f"Total Optimizations: {len(all_trials) * len(EVAL_SEEDS)}")
print("=" * 70)

start_time = time.time()
all_results = []

for i, trial in enumerate(all_trials):
    try:
        result = run_single_trial(trial)
        all_results.append(result)
        
        # Fortschritt anzeigen
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        remaining = (len(all_trials) - i - 1) / rate if rate > 0 else 0
        
        status_icon = "✓" if result.get('status') == 'completed' else "✗"
        print(f"[{i+1:3d}/{len(all_trials)}] {status_icon} {trial['algorithm']:10s} | "
              f"pop={trial['pop_size']:3d} | "
              f"HV={result.get('hypervolume_mean', 0):.4f} ± {result.get('hypervolume_std', 0):.4f} | "
              f"Time: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min")
              
    except Exception as e:
        print(f"[{i+1:3d}/{len(all_trials)}] ✗ Trial failed: {e}")
        all_results.append({
            'trial_id': trial['trial_id'],
            'algorithm': trial['algorithm'],
            'pop_size': trial['pop_size'],
            'eta_crossover': trial['eta_crossover'],
            'eta_mutation': trial['eta_mutation'],
            'crossover_prob': trial['crossover_prob'],
            'status': 'failed',
            'error': str(e)
        })

total_time = time.time() - start_time

print("\n" + "=" * 70)
print("TUNING COMPLETED!")
print("=" * 70)
print(f"Total Runtime: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
completed_count = len([r for r in all_results if r.get('status') == 'completed'])
failed_count = len([r for r in all_results if r.get('status') == 'failed'])
print(f"Completed: {completed_count}")
print(f"Failed: {failed_count}")
print(f"Success Rate: {completed_count/len(all_results)*100:.1f}%")

---

## 6. Ergebnis-Analyse

In [ ]:
# DataFrame erstellen
results_df = pd.DataFrame(all_results)

# Prüfen ob Ergebnisse vorhanden sind
if len(results_df) == 0:
    print("FEHLER: Keine Ergebnisse vorhanden!")
    completed_df = pd.DataFrame()
    best_configs = {}
elif 'status' not in results_df.columns:
    print("WARNUNG: 'status' Spalte fehlt.")
    completed_df = results_df.copy()
else:
    completed_df = results_df[results_df['status'] == 'completed'].copy()

print(f"Completed Trials: {len(completed_df)} / {len(results_df)}")
print(f"\nStatistiken pro Algorithmus:")
if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    stats = completed_df.groupby('algorithm')['hypervolume_mean'].agg(['count', 'mean', 'std', 'min', 'max'])
    print(stats.to_string())

In [ ]:
# Beste Konfiguration pro Algorithmus
best_configs = {}

if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    print("\n" + "=" * 70)
    print("BEST CONFIGURATION PER ALGORITHM")
    print("=" * 70)

    for algo in ALGORITHMS:
        algo_df = completed_df[completed_df['algorithm'] == algo]
        if len(algo_df) > 0:
            best_idx = algo_df['hypervolume_mean'].idxmax()
            best = algo_df.loc[best_idx].to_dict()
            best_configs[algo] = best
            
            print(f"\n{algo}:")
            print(f"  Hypervolume: {best['hypervolume_mean']:.4f} ± {best.get('hypervolume_std', 0):.4f}")
            print(f"  Range: [{best.get('hypervolume_min', 0):.4f}, {best.get('hypervolume_max', 0):.4f}]")
            print(f"  pop_size: {best['pop_size']}")
            print(f"  eta_crossover: {best['eta_crossover']:.2f}")
            print(f"  eta_mutation: {best['eta_mutation']:.2f}")
            print(f"  crossover_prob: {best['crossover_prob']:.3f}")
            if 'n_partitions' in best and pd.notna(best.get('n_partitions')):
                print(f"  n_partitions: {int(best['n_partitions'])}")
        else:
            print(f"\n{algo}: Keine erfolgreichen Trials")
else:
    print("Keine erfolgreichen Trials vorhanden!")

In [ ]:
# Top 5 pro Algorithmus
if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    print("\n" + "=" * 70)
    print("TOP 5 CONFIGURATIONS PER ALGORITHM")
    print("=" * 70)
    
    for algo in ALGORITHMS:
        algo_df = completed_df[completed_df['algorithm'] == algo].copy()
        if len(algo_df) > 0:
            top5 = algo_df.nlargest(5, 'hypervolume_mean')[['pop_size', 'eta_crossover', 'eta_mutation', 'crossover_prob', 'hypervolume_mean', 'hypervolume_std']]
            print(f"\n{algo}:")
            print(top5.to_string(index=False))

---

## 7. Visualisierung

In [ ]:
# Boxplot-Vergleich
if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    colors = {'NSGA-II': '#2E86AB', 'SMS-EMOA': '#E94F37', 'NSGA-III': '#3DDC97'}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Hypervolume Distribution
    data_for_plot = [completed_df[completed_df['algorithm'] == algo]['hypervolume_mean'].values 
                     for algo in ALGORITHMS if len(completed_df[completed_df['algorithm'] == algo]) > 0]
    labels = [algo for algo in ALGORITHMS if len(completed_df[completed_df['algorithm'] == algo]) > 0]

    if len(data_for_plot) > 0:
        bp = axes[0].boxplot(data_for_plot, labels=labels, patch_artist=True,
                        flierprops={'marker': 'x', 'markersize': 5})
        for patch, algo in zip(bp['boxes'], labels):
            patch.set_facecolor(colors[algo])
            patch.set_alpha(0.7)
        axes[0].set_ylabel('Hypervolume')
        axes[0].set_title('Hypervolume Distribution by Algorithm')
        axes[0].grid(True, alpha=0.3)

    # Violin Plot
    if len(labels) > 0:
        sns.violinplot(data=completed_df, x='algorithm', y='hypervolume_mean', 
                       palette=colors, ax=axes[1])
        axes[1].set_xlabel('Algorithm')
        axes[1].set_ylabel('Hypervolume')
        axes[1].set_title('Hypervolume Distribution (Violin)')

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'full_tuning_comparison.png', dpi=150)
    plt.show()
else:
    print("Keine Daten für Visualisierung vorhanden")

In [ ]:
# Parameter Impact Analysis
if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = {'NSGA-II': '#2E86AB', 'SMS-EMOA': '#E94F37', 'NSGA-III': '#3DDC97'}
    
    params_to_plot = ['pop_size', 'eta_crossover', 'eta_mutation', 'crossover_prob']
    titles = ['Population Size', 'SBX Eta (Crossover)', 'PM Eta (Mutation)', 'Crossover Probability']
    
    for idx, (param, title) in enumerate(zip(params_to_plot, titles)):
        ax = axes[idx // 2, idx % 2]
        
        for algo in ALGORITHMS:
            algo_df = completed_df[completed_df['algorithm'] == algo]
            if len(algo_df) > 0 and param in algo_df.columns:
                ax.scatter(algo_df[param], algo_df['hypervolume_mean'], 
                          c=colors[algo], alpha=0.6, s=40, label=algo)
        
        ax.set_xlabel(title)
        ax.set_ylabel('Hypervolume')
        ax.set_title(f'Impact of {title}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'full_tuning_parameter_impact.png', dpi=150)
    plt.show()
else:
    print("Keine Daten für Parameter Impact Analyse")

In [ ]:
# Heatmap: pop_size vs eta_crossover
if len(completed_df) > 0 and 'hypervolume_mean' in completed_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, algo in enumerate(ALGORITHMS):
        algo_df = completed_df[completed_df['algorithm'] == algo]
        if len(algo_df) > 5:
            # Binning für Heatmap
            algo_df = algo_df.copy()
            algo_df['pop_bin'] = pd.cut(algo_df['pop_size'], bins=5)
            algo_df['eta_bin'] = pd.cut(algo_df['eta_crossover'], bins=5)
            
            pivot = algo_df.pivot_table(values='hypervolume_mean', 
                                        index='eta_bin', 
                                        columns='pop_bin', 
                                        aggfunc='mean')
            
            sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[idx])
            axes[idx].set_title(f'{algo}: pop_size vs eta_crossover')
            axes[idx].set_xlabel('Population Size')
            axes[idx].set_ylabel('SBX Eta')
        else:
            axes[idx].text(0.5, 0.5, f'{algo}\nNicht genug Daten', 
                          ha='center', va='center', fontsize=14)
            axes[idx].set_title(algo)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'full_tuning_heatmap.png', dpi=150)
    plt.show()
else:
    print("Keine Daten für Heatmap")

---

## 8. Ergebnisse speichern

In [ ]:
# Alle Ergebnisse speichern
if len(all_results) > 0:
    results_export = {
        'config': {
            'K': K, 'L': L,
            'N_TRIALS': N_TRIALS,
            'N_GEN': N_GEN,
            'R_SIMULATIONS': R_SIMULATIONS,
            'EVAL_SEEDS': EVAL_SEEDS,
            'ALGORITHMS': ALGORITHMS,
            'SEARCH_SPACE': {k: list(v) if isinstance(v, tuple) else v for k, v in SEARCH_SPACE.items()},
            'total_runtime_minutes': total_time / 60
        },
        'all_trials': all_results,
        'best_per_algorithm': {algo: {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) 
                                       for k, v in config.items()} 
                              for algo, config in best_configs.items()}
    }

    with open(RESULTS_DIR / 'full_tuning_results.json', 'w') as f:
        json.dump(results_export, f, indent=2, default=str)

    print(f"Results saved to: {RESULTS_DIR / 'full_tuning_results.json'}")
else:
    print("Keine Ergebnisse zum Speichern vorhanden")

In [ ]:
# Beste Hyperparameter exportieren
if len(best_configs) > 0:
    best_params = {}

    for algo, config in best_configs.items():
        best_params[algo] = {
            'pop_size': int(config['pop_size']),
            'eta_crossover': float(config['eta_crossover']),
            'eta_mutation': float(config['eta_mutation']),
            'crossover_prob': float(config['crossover_prob']),
            'hypervolume_mean': float(config['hypervolume_mean']),
            'hypervolume_std': float(config.get('hypervolume_std', 0))
        }
        if 'n_partitions' in config and pd.notna(config.get('n_partitions')):
            best_params[algo]['n_partitions'] = int(config['n_partitions'])

    with open(RESULTS_DIR / 'best_hyperparameters.json', 'w') as f:
        json.dump(best_params, f, indent=2)

    print(f"Best parameters saved to: {RESULTS_DIR / 'best_hyperparameters.json'}")
    print("\n" + "=" * 70)
    print("FINAL BEST HYPERPARAMETERS")
    print("=" * 70)
    print(json.dumps(best_params, indent=2))
else:
    print("Keine besten Hyperparameter vorhanden")

In [ ]:
# CSV Export für weitere Analyse
if len(completed_df) > 0:
    completed_df.to_csv(RESULTS_DIR / 'full_tuning_trials.csv', index=False)
    print(f"CSV saved to: {RESULTS_DIR / 'full_tuning_trials.csv'}")

---

## 9. Zusammenfassung

### Verwendung der optimierten Parameter

```python
import json
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM

with open('../results/best_hyperparameters.json', 'r') as f:
    best = json.load(f)

# NSGA-II mit optimierten Parametern
algorithm = NSGA2(
    pop_size=best['NSGA-II']['pop_size'],
    crossover=SBX(prob=best['NSGA-II']['crossover_prob'], 
                  eta=best['NSGA-II']['eta_crossover']),
    mutation=PM(eta=best['NSGA-II']['eta_mutation'])
)
```